# Make COSMoS Model — the COSMoS authored model graph

Reads the COSMoS base graph (`base/cosmos/cosmos.ttl`) and, as a rule document, COSMoS's own LinkML model files (`model/cosmos_bc_model.yaml`, `model/cosmos_sdtm_model.yaml`, pinned in `Reference_Versions.json`; put them in `base/cosmos/downloads/`). Nothing else is read.

1. the mapping `schema/cosmos_model.yaml`, checked against the model files and the base module (`cfa.model.load_model_map`);
2. the three COSMoS tables, rebuilt from the base graph;
3. the model graph: COSMoS's classes and slots, with the IRIs as COSMoS's LinkML expands them; every value is the base cell as published (a plain literal). BiomedicalConcept, DataElementConcept and SDTMVariable are base row nodes; SDTMGroup, Coding, CodeList, AssignedTerm and RelationShip are nodes by rule (`schema/cosmos_model.yaml`);
4. reverse check: the BC table and the Dataset Specialization table rebuilt from the model graph must equal the base tables. A cell split at ";" is compared as its set of parts, as RDF keeps no order and no repeated part;
5. the cells that repeat a part (for COSMoS);
6. writes `cosmos-model.ttl` (longturtle) and its hash, `COSMoS_Model_Check.xlsx`, `Provenance_COSMoS_Model.json`.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import yaml
from rdflib import RDF, Graph, URIRef

from cfa import SCHEMA_DIR
from cfa.graph import graph_sha256, write_longturtle
from cfa.model import compare, flatten, load_model_map, model_graph, model_graph_iri, repeated_parts, split_columns
from cfa.pins import checked_source_file, file_sha256, reference_versions
from cfa.provenance import TOOLS, flat
from cfa.read import read_xlsx_table
from cfa.rule import class_iri, key_columns, load_module, row_iri_prefix
from cfa.view import TEXT_HASH_METHOD, text_sha256, view_frame, write_view
from importlib.metadata import version

MAP = SCHEMA_DIR / 'cosmos_model.yaml'
MODULE = SCHEMA_DIR / 'cosmos.yaml'
BASE = REPO_ROOT / 'base' / 'cosmos' / 'cosmos.ttl'
OUT = REPO_ROOT / 'linked'
SOURCE = 'cosmos'

base_sv = load_module(MODULE)
model_files = yaml.safe_load(MAP.read_text(encoding='utf-8'))['model_files']
model_paths = {k: checked_source_file(SOURCE, name) for k, name in model_files.items()}
mm = load_model_map(MAP, model_paths, base_sv)
for name, c in mm['classes'].items():
    print(f"{name}: {c['iri']}")

## 2. The COSMoS tables, from the base graph

In [ ]:
base = Graph().parse(BASE)
TABLES = list(base_sv.all_classes())
frames = {t: view_frame(base, base_sv, t) for t in TABLES}
keys = {t: key_columns(base_sv, t) for t in TABLES}
base_rows = set()
for t in TABLES:
    base_rows |= set(base.subjects(RDF.type, URIRef(class_iri(SOURCE, t))))
for t, df in frames.items():
    print(t, df.shape)

## 3. The model graph

In [ ]:
g = model_graph(mm, frames, base_rows)
print('triples', len(g))
for name, c in mm['classes'].items():
    print(f"{name}: {len(set(g.subjects(RDF.type, c['iri'])))} nodes")

## 4. Reverse check

Each table that the model covers, rebuilt from the model graph alone, against the base table. The hierarchy table has no slot in the model; it stays in base only.

In [ ]:
sep = mm['separator']
checked = {}
for t in sorted({c['table'] for c in mm['classes'].values()}):
    rebuilt = flatten(mm, g, t)
    diff = compare(rebuilt, frames[t], keys[t], split_columns(mm, t), sep)
    assert diff == {}, f'{t}: {diff}'
    checked[t] = {'rows': len(rebuilt), 'columns': len(rebuilt.columns), 'split_columns_compared_as_sets': split_columns(mm, t)}
    print(t, 'equal:', checked[t])

## 5. Cells that repeat a part

A cell split at ";" whose parts repeat: the model graph holds the part once. The published cell stays in base.

In [ ]:
repeated = repeated_parts(mm, {t: frames[t] for t in checked}, keys)
print(len(repeated), 'cells')
repeated

## 6. Write the graph, the check view, Provenance

In [ ]:
PREFIXES = {
    'cosmos_bc': 'https://www.cdisc.org/cosmos/biomedical_concept_v1.0',
    'cosmos_sdtm': 'https://www.cdisc.org/cosmos/sdtm_v1.0/',
}
for i, t in enumerate(TABLES, 1):
    PREFIXES[f'r{i}'] = row_iri_prefix(SOURCE, t)
path = OUT / 'cosmos-model.ttl'
write_longturtle(g, path, PREFIXES)
back = Graph().parse(path)
assert graph_sha256(back) == graph_sha256(g)
GRAPH_HASH = graph_sha256(g)
print(path.name, len(g), GRAPH_HASH)

schema_rows = []
for name, c in mm['classes'].items():
    for slot, spec in c['slots'].items():
        schema_rows.append({'Class': name, 'Class IRI': str(c['iri']), 'Slot': slot, 'Slot IRI': str(spec['iri']),
                            'Table': c['table'], 'Column': spec.get('column', ''),
                            'Split': 'Yes' if spec.get('split') else '', 'Object': spec.get('object', ''),
                            'Node': c['node']})
schema_df = pd.DataFrame(schema_rows, dtype=str)

KEYS = ['Table', 'Row', 'Column']
CHECK_HASH = text_sha256(repeated, KEYS)
pin = reference_versions()[SOURCE]
prov = {
    'layer': 'linked',
    'graph': model_graph_iri(),
    'inputs': {BASE.relative_to(REPO_ROOT).as_posix(): file_sha256(BASE)},
    'rule_documents': {f"{pin['repository']}/blob/{pin['commit']}/{name}": pin['files'][name]['sha256'] for name in model_files.values()},
    'schema': {p.relative_to(REPO_ROOT).as_posix(): file_sha256(p) for p in (MAP, MODULE)},
    'file': path.name,
    'triples': len(g),
    'graph_sha256': GRAPH_HASH,
    'graph_sha256_method': 'SHA-256 of the canonical N-Triples (RDFC-1.0; no blank nodes)',
    'reverse_check': checked,
    'repeated_parts_text_sha256': CHECK_HASH,
    'text_sha256_method': TEXT_HASH_METHOD,
    'tools': {t: version(t) for t in TOOLS},
}
(OUT / 'Provenance_COSMoS_Model.json').write_text(json.dumps(prov, indent=1, ensure_ascii=False) + '\n', encoding='utf-8')

readme = [
    'COSMoS_Model_Check: the COSMoS authored model graph (cosmos-model.ttl), built from the COSMoS base graph with the classes and slots of COSMoS\'s own LinkML model (cdisc-org/COSMoS, model/, pinned).',
    'Sheet Repeated_parts: cells split at ";" in which a part repeats. The model graph holds that part once; the published cell stays in the base graph. Value is the cell as published.',
    'Sheet Schema: per slot, the class and slot IRI as COSMoS\'s LinkML expands them, and the base column that fills it (schema/cosmos_model.yaml). Provenance: the inputs, the rule documents and the hashes.',
    'cdisc-for-all: https://github.com/kerfors/cdisc-for-all',
]
xlsx = OUT / 'COSMoS_Model_Check.xlsx'
write_view(xlsx, readme, {'Repeated_parts': repeated}, schema_df, flat(prov))
assert text_sha256(read_xlsx_table(xlsx, 'Repeated_parts'), KEYS) == CHECK_HASH
print(xlsx.name, 'read back: text hash equal', CHECK_HASH)